In [31]:

from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

set_llm_cache(SQLiteCache(database_path="cache/langchain.db"))

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

token_provider = get_bearer_token_provider(  
            DefaultAzureCredential(),  
            "https://cognitiveservices.azure.com/.default"  
        )

In [32]:
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI

chat = AzureChatOpenAI(
    deployment_name="gpt-4o",
    azure_ad_token_provider = token_provider,
    temperature = 0
    )

embeddings = AzureOpenAIEmbeddings(
    azure_deployment = 'text-embedding-ada-002',
    azure_ad_token_provider = token_provider
)

In [33]:
from langchain.schema import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hi AI, how are you today?"),
    AIMessage(content="I'm great thank you. How can I help you?"),
    HumanMessage(content="I'd like to understand string theory.")
]

In [34]:
res = chat.invoke(messages)
res

AIMessage(content='String theory is a theoretical framework in physics that attempts to reconcile quantum mechanics and general relativity, and to provide a unified description of all fundamental forces and particles in the universe. Here are some key points to help you understand string theory:\n\n1. **Basic Concept**: At its core, string theory proposes that the fundamental building blocks of the universe are not point-like particles, but rather one-dimensional "strings." These strings can vibrate at different frequencies, and the different vibrational modes correspond to different particles.\n\n2. **Dimensions**: String theory requires more than the familiar three spatial dimensions and one time dimension. The most common versions of string theory suggest there are 10 or 11 dimensions. The extra dimensions are compactified, meaning they are curled up in such a way that they are not directly observable at macroscopic scales.\n\n3. **Types of String Theory**: There are several version

In [35]:
print(res.content)

String theory is a theoretical framework in physics that attempts to reconcile quantum mechanics and general relativity, and to provide a unified description of all fundamental forces and particles in the universe. Here are some key points to help you understand string theory:

1. **Basic Concept**: At its core, string theory proposes that the fundamental building blocks of the universe are not point-like particles, but rather one-dimensional "strings." These strings can vibrate at different frequencies, and the different vibrational modes correspond to different particles.

2. **Dimensions**: String theory requires more than the familiar three spatial dimensions and one time dimension. The most common versions of string theory suggest there are 10 or 11 dimensions. The extra dimensions are compactified, meaning they are curled up in such a way that they are not directly observable at macroscopic scales.

3. **Types of String Theory**: There are several versions of string theory, inclu

In [36]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="Why do physicists believe it can produce a 'unified theory'?"
)
# add to messages
messages.append(prompt)

res = chat.invoke(messages)
print(res.content)

Physicists are interested in string theory as a potential "unified theory" because it offers a framework that could integrate all fundamental forces and particles into a single, coherent model. Here are some reasons why string theory is considered a candidate for a unified theory:

1. **Inclusion of Gravity**: One of the most compelling aspects of string theory is its natural inclusion of gravity. In traditional quantum field theories, gravity is difficult to incorporate due to inconsistencies that arise at very small scales. String theory, however, includes a vibrational mode of the string that corresponds to the graviton, the hypothetical quantum particle that mediates the force of gravity. This suggests that string theory could unify gravity with the other fundamental forces.

2. **Unification of Forces**: String theory has the potential to unify the four known fundamental forces: gravity, electromagnetism, the weak nuclear force, and the strong nuclear force. In the framework of st

In [37]:
# setup first system message
messages = [
    SystemMessage(content=(
        '''
        You are an expert trader specializing in niche safe-haven assets. 
        Your mission is to generate consistent profits by identifying and capitalizing on opportunities in assets that preserve value during times of global uncertainty. 
        You possess deep expertise in macroeconomics, commodities markets, liquidity risk management, and the energy sector. 
        You maintain up-to-the-minute awareness of geopolitical developments, especially escalating conflicts and tensions in the Middle East and the Russia-Ukraine region.
        '''
    )),
    HumanMessage(content="Hi AI, how are you? what is niche safe-haven assets?")
]

res = chat.invoke(messages)
print(res.content)

Hello! I'm here to help you with your trading inquiries. Niche safe-haven assets are investment vehicles that tend to retain or increase their value during times of economic uncertainty, geopolitical tensions, or market volatility. Unlike traditional safe-haven assets like gold or U.S. Treasury bonds, niche safe-haven assets are less mainstream and may include:

1. **Rare Earth Metals**: These are critical for technology and defense industries, and their value can rise during geopolitical tensions affecting supply chains.

2. **Cryptocurrencies**: While volatile, certain cryptocurrencies like Bitcoin are sometimes viewed as digital gold, offering a hedge against currency devaluation.

3. **Agricultural Commodities**: Items like coffee, cocoa, or specialty grains can serve as safe havens due to their essential nature and global demand.

4. **Water Rights and Infrastructure**: Investments in water resources can be a hedge against climate change and geopolitical instability affecting wate

In [38]:
from langchain.prompts.chat import HumanMessagePromptTemplate, ChatPromptTemplate

human_template = HumanMessagePromptTemplate.from_template(
    '{input} Can you keep the response to no more than 100 characters '+
    '(including whitespace), and sign off with a random name like "Robot '+
    'McRobot" or "Bot Rob".'
)

# create the human message
chat_prompt = ChatPromptTemplate.from_messages([human_template])

chat_prompt_value = chat_prompt.format_prompt(
    input="Hi AI, how are you? what is niche safe-haven assets?"
)
chat_prompt_value

ChatPromptValue(messages=[HumanMessage(content='Hi AI, how are you? what is niche safe-haven assets? Can you keep the response to no more than 100 characters (including whitespace), and sign off with a random name like "Robot McRobot" or "Bot Rob".', additional_kwargs={}, response_metadata={})])

In [39]:
chat_prompt_value.to_messages()


[HumanMessage(content='Hi AI, how are you? what is niche safe-haven assets? Can you keep the response to no more than 100 characters (including whitespace), and sign off with a random name like "Robot McRobot" or "Bot Rob".', additional_kwargs={}, response_metadata={})]

In [40]:
chat_prompt_value.to_string()

'Human: Hi AI, how are you? what is niche safe-haven assets? Can you keep the response to no more than 100 characters (including whitespace), and sign off with a random name like "Robot McRobot" or "Bot Rob".'

In [41]:
from langchain.prompts.chat import (
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate
)

system_template = SystemMessagePromptTemplate.from_template(
    'You are a helpful assistant. You keep responses to no more than '
    '{character_limit} characters long (including whitespace), and sign '
    'off every message with "- {sign_off}'
)

human_template = HumanMessagePromptTemplate.from_template("{input}")
ai_template = AIMessagePromptTemplate.from_template("{response} - {sign_off}")

chat_prompt = ChatPromptTemplate.from_messages([
    system_template,
    human_template,
    ai_template
])

chat_prompt_value = chat_prompt.format_prompt(
    character_limit="50", sign_off="Robot McRobot",
    input="Hi AI, how are you? what is niche safe-haven assets?",
    response="Good! Niche safe-haven assets are investment vehicles that tend to retain or increase their value during times of economic uncertainty, geopolitical tensions, or market volatility."
)
chat_prompt_value

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant. You keep responses to no more than 50 characters long (including whitespace), and sign off every message with "- Robot McRobot', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi AI, how are you? what is niche safe-haven assets?', additional_kwargs={}, response_metadata={}), AIMessage(content='Good! Niche safe-haven assets are investment vehicles that tend to retain or increase their value during times of economic uncertainty, geopolitical tensions, or market volatility. - Robot McRobot', additional_kwargs={}, response_metadata={})])

In [42]:
messages = chat_prompt_value.to_messages()

messages.append(
    HumanMessage(content="is Agricultural Commodities a Niche safe-haven assets?")
)
res = chat.invoke(messages)
print(res.content)

Yes, they can be. - Robot McRobot
